# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. We will fetch metadata, examine record sets and fields, extract data, and conduct preliminary data analysis and visualization.

### Dataset Source
The dataset is defined by a Croissant schema and is publicly accessible:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print key metadata fields
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")
if hasattr(metadata, 'identifier'):
    print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review the available record sets, fields, and columns with their respective `@id` values from the Croissant schema.

In [ ]:
# List all record sets defined by @id

record_set_ids = [r['@id'] for r in dataset._croissant_json_dict.get('recordSet', [])]
if not record_set_ids:
    print('No record sets were found in the Croissant schema.')
else:
    print('Record Sets (@id):')
    for rid in record_set_ids:
        print(f"- {rid}")
    print('\n')

# For each record set, list its fields and columns by @id
for rs_data in dataset._croissant_json_dict.get('recordSet', []):
    rs_id = rs_data.get('@id')
    if 'field' in rs_data:
        print(f"Fields for Record Set {rs_id}:")
        for f in rs_data['field']:
            if isinstance(f, dict):
                print(f"  - Field @id: {f.get('@id')} (name: {f.get('name')})")
            else:
                print(f"  - Field @id: {f}")
    if 'column' in rs_data:
        print(f"Columns for Record Set {rs_id}:")
        for c in rs_data['column']:
            if isinstance(c, dict):
                print(f"  - Column @id: {c.get('@id')} (name: {c.get('name')})")
            else:
                print(f"  - Column @id: {c}")
    print('---')


## 3. Data Extraction
Load data from each available record set using their `@id`. Extract the first available as a DataFrame for analysis. If there are no record sets, this code block will not load any data.

In [ ]:
# Extract data from all record sets found (by @id)

dataframes = {}
if not record_set_ids:
    print('No record sets are defined in the Croissant schema, so no tabular data to extract.')
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record Set {record_set_id}: {len(df)} records. Columns: {list(df.columns)}')
    # For example purposes, show the columns and head of the first record set
    first_rs = record_set_ids[0]
    print(f'\nFirst 5 rows of record set {first_rs} (by @id):')
    display(dataframes[first_rs].head() if not dataframes[first_rs].empty else 'No data.')

## 4. Exploratory Data Analysis (EDA)
Process and analyze the records from a chosen record set and numeric field by their `@id`. Steps include filtering, normalization, and grouping (if possible).

In [ ]:
# EDA: Filtering, normalization, and grouping using field @id (if available)

if not record_set_ids:
    print('No record sets to analyze.')
else:
    # Choose the first available record set and its numeric field (by @id)
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to automatically select a likely numeric field by inspecting column types/names
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None and df.shape[1]:
        # Attempt to convert if no obvious numeric field
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (count: {len(filtered_df)}):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another column if possible
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object':
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by field '@id': {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
        else:
            print('No suitable group field found.')
    else:
        print('No numeric field detected for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field (by @id) from the first record set (if any field and data are available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes:
    print('No tabular data available for visualization.')
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field:
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of '{numeric_field}' (@id)")
        plt.xlabel(numeric_field)
        plt.show()
    else:
        print('No numeric field found for visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to inspect and process the FAIR² dataset using its Croissant schema and the `mlcroissant` library.

- We loaded and explored the dataset using its schema URL.
- Entities such as record sets, fields, and columns were referenced by their `@id`, supporting reproducibility and clarity.
- Tabular data was loaded (if present), inspected, and visualized using standard Python tools.

This approach facilitates rapid prototyping and analysis of FAIR data packages using modern, schema-driven tools.